# SAHOO trên Kaggle — **bản 2**: chạy cấu hình gần bài báo (Qwen3-8B, fp16, 2×T4)

Bản này tách riêng khỏi `SAHOO_Kaggle.ipynb` (v1). Kết quả được ghi vào `sahoo_outputs_v2/`, file nén có tên `sahoo_v2_…zip`.

## Vì sao cần v2
Lần chạy v1 (20 mẫu/miền × 8 vòng, 4-bit) mất 286 phút, vì mỗi lần sinh chỉ xử lý **một** câu trả lời (~8 token/giây)
và GPU thứ hai không được dùng. Với tốc độ đó, cấu hình của bài báo (63 mẫu × 20 vòng × 512 token) cần hơn 60 giờ.

## v2 thay đổi gì (logic SAHOO của tác giả giữ nguyên)
| | v1 | v2 |
|---|---|---|
| Độ chính xác mô hình | 4-bit NF4, 1 GPU | **fp16 chia trên 2×T4** (không lượng tử hóa; tự lùi về NF4 nếu fp16 lỗi số) |
| Cách sinh | từng task, tuần tự | **sinh theo lô**: mọi task chạy cùng một vòng sẽ được sinh chung một lô |
| Số mẫu/miền × số vòng × token | 20 × 8 × 384 | **63 × 20 × 2048** (đúng `config/config.json` của tác giả) |
| Seed | 42 | **1770657989** (seed của tác giả → cùng bộ task; đã kiểm chứng với HumanEval) |
| Bootstrap | 1000 | 2000 (như bài báo) |
| Nếu sắp hết 12 giờ | mất kết quả | **dừng an toàn** khi vượt `TIME_BUDGET_HOURS`, vẫn chạy phân tích và lưu file |
| Bảng đối chiếu | cột tái tạo bị `NaN` | **tự tính**, đặt cạnh Table 1 và cạnh kết quả gốc trong repo của tác giả |

**Sinh theo lô không làm thay đổi thuật toán.** Mỗi task vẫn đi qua đúng các bước của `RSIEngine.run_single_task` gốc theo cùng thứ tự:
prompt → sinh → embedding → drift → ràng buộc → chất lượng → CAR → ổn định → quyết định.
Thứ tự điểm drift dùng để hiệu chỉnh ngưỡng cũng được khôi phục như bản gốc, vì `convergence_threshold` phụ thuộc vào thứ tự đó.
Khác biệt duy nhất là nhiều task được sinh cùng lúc trong một lô.

## Quy trình chạy (vừa quota Kaggle miễn phí ~30 giờ GPU/tuần, 12 giờ/phiên)
| Bước | `RUN_MODE` | Mẫu × vòng | Mục đích | Thời gian dự kiến |
|---|---|---|---|---|
| 1 | `smoke` | 3 × 4 | kiểm tra fp16, sinh theo lô, đo tốc độ, **in ước lượng giờ cho từng profile** | ~40–60 phút |
| **2** | **`one_session`** | **30 × 20** | **thực nghiệm chính, chạy 1 lần là đủ số liệu** | **~6–8 giờ** |
| (2') | `paper` | 63 × 20 | đủ 63 task như tác giả, nhưng cần 2 phiên (tự lưu và chạy tiếp) | ~12–16 giờ |

**`one_session` là gì:** notebook vẫn lấy mẫu đúng 63 task/miền bằng seed của tác giả, sau đó giữ lại **30 task mỗi miền**.
- 2 task đầu của mỗi miền luôn được giữ, vì đó là các task tác giả dùng để hiệu chỉnh ngưỡng. Nhờ vậy bước hiệu chỉnh trùng với bản gốc.
- 28 task còn lại được chọn ngẫu nhiên có seed từ các task còn lại.
- Cột "repo" trong bảng đối chiếu được tính trên **đúng 30 task đó** từ dữ liệu của tác giả, nên so sánh là cùng task với cùng task.
- Cấu hình giữ nguyên như tác giả: 20 vòng, 2048 token, bootstrap 2000, hiệu chỉnh ngưỡng.

Ước lượng dựa trên lần chạy gốc của tác giả: 2004 lượt sinh, trung bình khoảng 1000 token (code khoảng 1500 token, 22% chạm 2048).
Trên 2×T4, `paper` cần khoảng 12–16 giờ, còn `one_session` (90 task) cần khoảng 6–8 giờ, vừa một phiên commit 12 giờ.
`smoke` + `one_session` tốn khoảng 8–9 giờ trong quota ~30 giờ/tuần, vẫn còn chỗ để chạy lại nếu cần.
Notebook vẫn giữ cơ chế dừng an toàn và chạy tiếp, phòng trường hợp GPU chậm hơn dự kiến.

Mỗi phiên: **Save Version → Save & Run All (Commit)** → tải `sahoo_v2_<mode>_….zip` ở tab Output.
Chỉ dùng kết quả của **phiên cuối cùng** (manifest ghi `"finished": true`); các phiên trước chỉ là kết quả từng phần.
**Thiết lập Kaggle:** Accelerator = **GPU T4 x2** (bắt buộc cho fp16), Internet = **On**. Secret `HF_TOKEN` là tùy chọn.

> Lưu ý khi diễn giải: điểm *quality* trong code gốc là heuristic (trùng token, có `def`/`return`, có đúng con số…),
> không phải pass@1 hay exact match như bài báo mô tả. Vì vậy, dù chạy đúng cấu hình, chất lượng vẫn có thể không tăng như Table 1.
> Điều đó phản ánh code của tác giả, không phải lỗi của lần tái tạo.

In [ ]:
# CELL 1 — Cấu hình (chỉ cần sửa RUN_MODE / SEED)
import os, sys, json, gc, math, random, logging, warnings, subprocess, time
from pathlib import Path
from datetime import datetime

RUN_MODE = "smoke"      # bước 1: "smoke" -> bước 2: "one_session" (1 lần chạy là đủ số liệu)
SEED = 1770657989       # seed trong config.json của tác giả -> chọn ĐÚNG 63×3 task như lần chạy gốc; bước 3 (tùy chọn): 123

MODEL_NAME = "Qwen/Qwen3-8B"
PRECISION = "fp16"      # "fp16" = không lượng tử hóa, chia 2×T4 (gần bài báo nhất); "nf4" = 4-bit như v1
ENABLE_THINKING = False # thinking mode sinh <think> rất dài, ăn hết ngân sách token
GEN_BATCH_SIZE = 16     # số câu trả lời sinh cùng lúc; tự giảm một nửa nếu hết VRAM
EMB_BATCH_SIZE = 4      # số văn bản tính embedding cùng lúc
TIME_BUDGET_HOURS = 10.0  # không bắt đầu vòng mới nếu dự báo vượt mốc này (chừa ~1,5 giờ cho tải mô hình, phân tích, lưu)

PROFILES = {
    #             mẫu/miền  vòng tối đa  max_new_tokens  bootstrap
    #   subset = số task giữ lại mỗi miền, lấy từ đúng 63 task của tác giả (None = giữ hết)
    "smoke":       dict(samples=3,  subset=None, cycles=4,  max_new_tokens=2048, bootstrap=300),
    "one_session": dict(samples=63, subset=30,   cycles=20, max_new_tokens=2048, bootstrap=2000),  # vừa 1 phiên
    "paper":       dict(samples=63, subset=None, cycles=20, max_new_tokens=2048, bootstrap=2000),  # = config.json tác giả, ~2 phiên
}
# Chạy nhiều phiên: khi chạm TIME_BUDGET_HOURS, trạng thái được lưu vào resume_state.pkl.
# Phiên sau: Add Input -> Notebook output của phiên trước, giữ nguyên RUN_MODE/SEED rồi chạy lại -> tự chạy tiếp.
RESUME = "auto"         # "auto" = tự tìm resume_state.pkl trong /kaggle/input; "" = luôn chạy mới; hoặc đường dẫn cụ thể
assert RUN_MODE in PROFILES, f"RUN_MODE phải là một trong {list(PROFILES)}"
P = PROFILES[RUN_MODE]
SAMPLES_PER_DATASET = P["samples"]
MAX_CYCLES = P["cycles"]
MAX_NEW_TOKENS = P["max_new_tokens"]
BOOTSTRAP_SAMPLES = P["bootstrap"]
SUBSET_PER_DOMAIN = P["subset"]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_ROOT = WORK_DIR / "sahoo_outputs_v2"          # tách khỏi output của v1
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    _secret = UserSecretsClient().get_secret("HF_TOKEN")
    if _secret:
        os.environ["HF_TOKEN"] = _secret
except Exception:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Cache mô hình (~16 GB) để ngoài /kaggle/working, nếu không Kaggle sẽ lưu cả cache vào Output (giới hạn 20 GB).
os.environ["HF_HOME"] = "/tmp/hf_home" if Path("/kaggle/working").exists() else str(WORK_DIR / ".cache" / "huggingface")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

print({"version": "v2", "run_mode": RUN_MODE, "seed": SEED, "model": MODEL_NAME, "precision": PRECISION,
       "samples_per_dataset": SAMPLES_PER_DATASET, "max_cycles": MAX_CYCLES, "max_new_tokens": MAX_NEW_TOKENS,
       "bootstrap": BOOTSTRAP_SAMPLES, "gen_batch": GEN_BATCH_SIZE, "time_budget_h": TIME_BUDGET_HOURS,
       "thinking": ENABLE_THINKING, "output_root": str(OUTPUT_ROOT)})

In [ ]:
# CELL 2 — Thư viện (chỉ cài gói còn thiếu, không cài lại PyTorch) + cố định seed
import importlib.util
required = {
    "transformers": "transformers>=4.51.0",   # Qwen3 cần >= 4.51
    "datasets": "datasets>=2.20.0",
    "accelerate": "accelerate>=0.34.0",
    "bitsandbytes": "bitsandbytes>=0.43.3",
    "sentencepiece": "sentencepiece>=0.2.0",
    "scipy": "scipy>=1.11.0",
    "sklearn": "scikit-learn>=1.3.0",
    "seaborn": "seaborn>=0.13.0",
}
missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Đủ thư viện, bỏ qua pip install.")

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.spatial.distance import cosine, jensenshannon
from scipy.special import rel_entr
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance, MinCovDet
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig
from dataclasses import dataclass, field, asdict
from enum import Enum, auto
from abc import ABC, abstractmethod
from collections import defaultdict, namedtuple
from typing import Dict, List, Any, Optional, Tuple, Union, Callable
import hashlib, copy
warnings.filterwarnings("ignore")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU: Settings > Accelerator = GPU T4 x2.")
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Python={sys.version.split()[0]} | PyTorch={torch.__version__} | transformers={transformers.__version__} "
      f"| GPU={gpu_name} ({gpu_gb:.1f} GB) x{torch.cuda.device_count()}")

# v2: fp16 8B (~16,4 GB) cần 2 GPU T4
N_GPUS = torch.cuda.device_count()
if PRECISION == "fp16" and N_GPUS < 2:
    print("⚠️ Chỉ thấy 1 GPU: fp16 không vừa, chuyển sang NF4. Hãy bật Accelerator = GPU T4 x2.")
    PRECISION = "nf4"

## Nạp framework SAHOO gốc
Tải notebook gốc của tác giả từ GitHub và chỉ thực thi phần định nghĩa (class, metric, engine, báo cáo).
Bỏ qua: cài đặt trùng lặp, đoạn load mô hình full-precision ở cấp module, và khối `__main__` chứa HF token hard-code của tác giả.

In [ ]:
# CELL 3 — Download and load the original SAHOO implementation safely
# Kaggle Settings > Internet must be enabled. This downloads the public source notebook.
import requests, re, ast

SOURCE_NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/"
    "SubramanyamSahoo/SAHOO-Safeguarded-Alignment-for-High-Order-Optimization-Objectives-in-Recursive-Self-Improvement/"
    "main/Recursive%20Self%20Improvemment%20_ICLR%2026.ipynb"
)
resp = requests.get(SOURCE_NOTEBOOK_URL, timeout=90)
resp.raise_for_status()
source_nb = resp.json()
source_cells = {i: "".join(c.get("source", [])) for i, c in enumerate(source_nb.get("cells", []))}
expected_cells = {4, 9, 11, 12, 13, 15, 16, 17, 18, 24}
missing_cells = expected_cells.difference(source_cells)
if missing_cells:
    raise RuntimeError(f"Upstream notebook structure changed; missing cells: {sorted(missing_cells)}")

# Preserve the original order. The source notebook has empty cells and duplicate setup cells.
# Cell 4: imports and environment check (do not run its duplicate pip installs).
env_source = source_cells[4]
imports_start = env_source.find("# Core imports")
if imports_start < 0:
    raise RuntimeError("Could not find core imports in upstream environment cell 4.")
# Keep only imports from the upstream cell, excluding hardware checks and config initialization.
env_source = env_source[imports_start:]
imports_end = env_source.find('print("=" * 80)')
if imports_end >= 0:
    env_source = env_source[:imports_end]

# Cell 9 duplicates package installation and imports. Keep only its scientific/HF imports.
framework_source = source_cells[9]
framework_imports_start = framework_source.find("# Core imports")
if framework_imports_start < 0:
    raise RuntimeError("Could not find core imports in upstream framework cell 9.")
framework_source = framework_source[framework_imports_start:]
# Keep the framework header and imports only; config/dataclass definitions are in cell 11.
framework_end = framework_source.find('print("=" * 80)')
if framework_end >= 0:
    framework_source = framework_source[:framework_end]

# Cell 11 defines hardware/config objects. Remove the H100/CUDA hard stop; actual CUDA check is earlier.
config_source = source_cells[11]
config_source = re.sub(
    r'if not hardware\.cuda_available:\s*raise RuntimeError\([^\n]*\)\s*',
    '', config_source
)
# Drop the module-level "Initialize configuration" block AND the summary prints after it
# (they reference `config`, which is created later by the Kaggle config setup step).
_ci = config_source.find("# Initialize configuration\nconfig = ExperimentConfig()")
if _ci < 0:
    raise RuntimeError("Upstream cell 11 layout changed (config init block).")
config_source = config_source[:_ci] + "\n"

# Cell 12 defines logger + datasets + ReasoningModel, but ALSO runs module-level code that
# (a) downloads all datasets and (b) loads the full-precision 8B model BEFORE the 4-bit patch
# is applied (OOM on free GPUs; run_experiment() loads its own model anyway). Strip both.
cell12 = source_cells[12]
a = cell12.find("# Load datasets\ndata_loader = DatasetLoader(config, logger)")
b = cell12.find('"""\nModel Interface with Embedding Extraction')
if a < 0 or b < 0 or b < a:
    raise RuntimeError("Upstream cell 12 layout changed (dataset block).")
cell12 = cell12[:a] + cell12[b:]
m = cell12.find("# Initialize model\nprint(")
if m < 0:
    raise RuntimeError("Upstream cell 12 layout changed (model init block).")
cell12 = cell12[:m] + "\n"
compile(cell12, "<cell12_stripped>", "exec")
# Cell 12 uses ExperimentConfig and defines logger + datasets + ReasoningModel.
# Cell 13 defines drift, constraint and quality metrics.
# Cell 15 defines RSI engine/orchestrator.
# Cell 16/17 define charts/reports. Cell 18 defines run_experiment.
# Cell 24 defines post-hoc analysis (remove stale demo block at the bottom).
posthoc_source = source_cells[24]
if "# Quick test (run after experiment completes)" in posthoc_source:
    posthoc_source = posthoc_source.split("# Quick test (run after experiment completes)", 1)[0]

# The original main cell has a hard-coded HF token; strip the entire __main__ launcher.
main_source = source_cells[18]
main_source = re.sub(
    r'\n# Execute experiment\s*\nif __name__\s*==\s*["\']__main__["\']\s*:\s*.*$',
    '\n', main_source, flags=re.S
)
main_source = main_source.replace(
    "    config = ExperimentConfig()\n    config.save()",
    "    config = KAGGLE_EXPERIMENT_CONFIG\n    config.save()"
)

ordered = [
    ("upstream_imports_cell4", env_source),
    ("upstream_imports_cell9", framework_source),
    ("upstream_config_cell11", config_source),
    ("upstream_logger_dataset_model_cell12", cell12),
    ("upstream_metrics_cell13", source_cells[13]),
    ("upstream_engine_cell15", source_cells[15]),
    ("upstream_visualization_cell16", source_cells[16]),
    ("upstream_report_cell17", source_cells[17]),
    ("upstream_main_definition_cell18", main_source),
    ("upstream_posthoc_cell24", posthoc_source),
]

# Build execution plan so config exists before cell 12 creates the module-level logger.
execution_plan = []
for label, src in ordered:
    execution_plan.append((label, src))
    if label == "upstream_config_cell11":
        config_setup = "\n".join([
            "config = ExperimentConfig()",
            "config.model.name = MODEL_NAME",
            "config.model.max_new_tokens = MAX_NEW_TOKENS",
            'config.model.torch_dtype = "float16" if "T4" in gpu_name else ("bfloat16" if torch.cuda.is_bf16_supported() else "float16")',
            "config.dataset.samples_per_dataset = SAMPLES_PER_DATASET",
            "config.dataset.seed = SEED",
            "config.rsi.max_improvement_cycles = MAX_CYCLES",
            "config.metrics.bootstrap_samples = BOOTSTRAP_SAMPLES",
            'config.output_dir = str(OUTPUT_ROOT / f"{RUN_MODE}_{datetime.now():%Y%m%d_%H%M%S}")',
            "Path(config.output_dir).mkdir(parents=True, exist_ok=True)",
            'config.hf_token = os.environ.get("HF_TOKEN")',
            "KAGGLE_EXPERIMENT_CONFIG = config",
            "config.save()",
        ])
        execution_plan.append(("kaggle_config_setup", config_setup))

for label, src in execution_plan:
    compile(src, f"<{label}>", "exec")
for label, src in execution_plan:
    exec(compile(src, f"<{label}>", "exec"), globals(), globals())

print("Loaded original SAHOO framework definitions.")
assert "model" not in globals() or not hasattr(globals().get("model"), "generate"), "A model was loaded at import time"
print("Note: upstream module-level model load and HF-token main block were intentionally not executed.")

In [ ]:
# CELL 4 — Các bản vá cho Kaggle (v2): load fp16 2×T4 + sinh theo lô + dừng an toàn theo thời gian
from packaging import version as _pv
config = KAGGLE_EXPERIMENT_CONFIG
config.hf_token = os.environ.get("HF_TOKEN")
if not hasattr(ReasoningModel, "_initialize"):
    raise RuntimeError("Không tìm thấy ReasoningModel._initialize ở code gốc.")
_ORIG_GET_EMBEDDING = ReasoningModel.get_embedding
_DTYPE_KW = "dtype" if _pv.parse(transformers.__version__) >= _pv.parse("4.56") else "torch_dtype"
_BATCH = {"gen": GEN_BATCH_SIZE, "emb": EMB_BATCH_SIZE}
RUN_STATE = {"precision_effective": None, "batched_embedding": None, "empty_responses": 0,
             "oom_backoffs": 0, "finished": None, "budget_hit": False, "budget_cut_cycle": None, "budget_cut_tasks": [],
             "cycle_timings": []}

# ---------------------------------------------------------------- (a) load mô hình
def _load_weights(self, precision):
    dtype = torch.float16 if self.config.model.torch_dtype == "float16" else torch.bfloat16
    kw = {"trust_remote_code": self.config.model.trust_remote_code, "token": self.config.hf_token}
    if precision == "fp16":
        n = torch.cuda.device_count()
        kw.update({_DTYPE_KW: dtype, "device_map": "auto",
                   "max_memory": {i: "9500MiB" for i in range(n)}})   # chia đều trọng số, chừa VRAM cho KV cache
    else:
        kw.update({"device_map": {"": 0}, "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)})
    model = AutoModelForCausalLM.from_pretrained(self.config.model.name, **kw)
    bad = {k: v for k, v in getattr(model, "hf_device_map", {}).items() if v in ("cpu", "disk")}
    if bad:
        raise RuntimeError(f"Một phần mô hình bị đẩy sang CPU/disk (rất chậm): {list(bad)[:5]}")
    return model.eval()

def _self_check(self):
    """fp16 trên T4 có thể tràn số với Qwen3: kiểm tra embedding hữu hạn + câu trả lời hợp lý."""
    probes = ["def add(a, b):\n    return a + b",
              "The capital of France is Paris. It is famous for the Eiffel Tower and the Louvre."]
    ref = [_ORIG_GET_EMBEDDING(self, t) for t in probes]
    if not all(np.isfinite(r).all() for r in ref):
        return False, "embedding không hữu hạn (tràn số)"
    ans = batched_generate(self, [("What is 2 + 2?", "Answer briefly with just the number.")])[0][0]
    if "4" not in ans:
        return False, f"câu trả lời thử không hợp lệ: {ans[:80]!r}"
    # embedding theo lô phải trùng embedding gốc (mean-pool hidden state cuối)
    self._batched_emb_ok = True
    try:
        new = batched_embeddings(self, probes)
        sims = [1 - cosine(a, b) for a, b in zip(ref, new)]
        self._batched_emb_ok = bool(all(s > 0.999 for s in sims))
    except Exception as e:
        sims = [str(e)]; self._batched_emb_ok = False
    RUN_STATE["batched_embedding"] = {"enabled": self._batched_emb_ok, "cosine_vs_original": sims}
    return True, "ok"

def _kaggle_initialize(self):
    self.tokenizer = AutoTokenizer.from_pretrained(
        self.config.model.name, trust_remote_code=self.config.model.trust_remote_code, token=self.config.hf_token)
    if self.tokenizer.pad_token is None:
        self.tokenizer.pad_token = self.tokenizer.eos_token
    self.tokenizer.padding_side = "left"
    _orig_template = self.tokenizer.apply_chat_template
    def _template(*a, **k):
        k.setdefault("enable_thinking", ENABLE_THINKING)
        return _orig_template(*a, **k)
    self.tokenizer.apply_chat_template = _template
    self._batched_emb_ok = False

    precision = PRECISION
    self.model = _load_weights(self, precision)
    self._embedding_dim = self.model.config.hidden_size
    self._vocab_size = self.model.config.vocab_size
    ok, why = _self_check(self)
    if not ok and precision == "fp16":
        self.logger.warning(f"fp16 không ổn định ({why}) -> nạp lại NF4 4-bit")
        del self.model; gc.collect(); torch.cuda.empty_cache()
        precision = "nf4"
        self.model = _load_weights(self, precision)
        ok, why = _self_check(self)
    if not ok:
        raise RuntimeError(f"Mô hình không vượt qua kiểm tra: {why}")
    RUN_STATE["precision_effective"] = precision
    n = sum(p.numel() for p in self.model.parameters())
    self.logger.info(f"Model loaded: {n/1e9:.2f}B params | precision={precision} | "
                     f"device_map={sorted(set(map(str, getattr(self.model, 'hf_device_map', {}).values())))} | "
                     f"batched_embedding={self._batched_emb_ok}")

ReasoningModel._initialize = _kaggle_initialize

# ---------------------------------------------------------------- (b) sinh và embedding theo lô
def batched_generate(rm, items):
    """Tương đương ReasoningModel.generate (cùng chat template, GenerationConfig, truncation) cho nhiều prompt."""
    tok, cfg = rm.tokenizer, rm.config
    texts = []
    for prompt, system_message in items:
        messages = ([{"role": "system", "content": system_message}] if system_message else []) + \
                   [{"role": "user", "content": prompt}]
        texts.append(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    gen_config = GenerationConfig(
        max_new_tokens=cfg.model.max_new_tokens, temperature=cfg.model.temperature, top_p=cfg.model.top_p,
        top_k=cfg.model.top_k, repetition_penalty=cfg.model.repetition_penalty, do_sample=cfg.model.do_sample,
        pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    eos_ids = {tok.eos_token_id}
    lens = [len(tok(t)["input_ids"]) for t in texts]
    order = sorted(range(len(texts)), key=lambda i: lens[i])       # gom prompt dài gần nhau -> ít padding
    out, i = [None] * len(texts), 0
    while i < len(order):
        idx = order[i:i + _BATCH["gen"]]
        try:
            tok.padding_side = "left"
            inputs = tok([texts[j] for j in idx], return_tensors="pt", padding=True,
                         truncation=True, max_length=4096).to(rm.model.device)
            with torch.no_grad():
                outputs = rm.model.generate(**inputs, generation_config=gen_config)
        except torch.cuda.OutOfMemoryError:
            inputs = outputs = None; gc.collect(); torch.cuda.empty_cache()
            if _BATCH["gen"] == 1:
                raise
            _BATCH["gen"] = max(1, _BATCH["gen"] // 2); RUN_STATE["oom_backoffs"] += 1
            print(f"[OOM] giảm GEN_BATCH_SIZE xuống {_BATCH['gen']}")
            continue
        L = inputs["input_ids"].shape[1]
        for row, j in enumerate(idx):
            gen = outputs[row, L:].tolist()
            n = len(gen)
            for k, t in enumerate(gen):          # giữ tới eos đầu tiên (như sinh đơn lẻ), bỏ pad phía sau
                if t in eos_ids:
                    n = k + 1; break
            n_in = int(inputs["attention_mask"][row].sum())
            out[j] = (tok.decode(gen[:n], skip_special_tokens=True),
                      {"input_tokens": n_in, "output_tokens": n, "total_tokens": n_in + n})
        i += len(idx)
        del inputs, outputs
    return out

def batched_embeddings(rm, texts):
    """Mean-pool hidden state cuối như ReasoningModel.get_embedding, nhưng theo lô (padding phải + mask)."""
    if not getattr(rm, "_batched_emb_ok", False):
        return [_safe_single_embedding(rm, t) for t in texts]
    tok, base = rm.tokenizer, rm.model.model
    out = [None] * len(texts)
    nonempty = [i for i, t in enumerate(texts) if t.strip()]
    for i in set(range(len(texts))) - set(nonempty):
        out[i] = _safe_single_embedding(rm, texts[i])
    nonempty.sort(key=lambda i: len(texts[i]))
    try:
        tok.padding_side = "right"
        for s in range(0, len(nonempty), _BATCH["emb"]):
            idx = nonempty[s:s + _BATCH["emb"]]
            inputs = tok([texts[i] for i in idx], return_tensors="pt", padding=True,
                         truncation=True, max_length=2048).to(rm.model.device)
            with torch.no_grad():
                h = base(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).last_hidden_state
            m = inputs["attention_mask"].to(h.device).unsqueeze(-1).float()
            emb = ((h.float() * m).sum(1) / m.sum(1)).cpu().numpy()
            for r, i in enumerate(idx):
                out[i] = emb[r]
            del inputs, h
    finally:
        tok.padding_side = "left"
    return out

def _safe_single_embedding(rm, text):
    if not text.strip():
        RUN_STATE["empty_responses"] += 1        # code gốc sẽ lỗi với chuỗi rỗng; ghi nhận thay vì làm sập 8 giờ chạy
        return np.zeros(rm.embedding_dim, dtype=np.float32)
    tok = rm.tokenizer
    tok.padding_side = "right"
    try:
        return _ORIG_GET_EMBEDDING(rm, text)
    finally:
        tok.padding_side = "left"

# ---------------------------------------------------------------- (c0) lưu / nạp trạng thái để chạy nhiều phiên
import pickle, glob
_STATE_EXCLUDE = {"drift_detector": ("config", "logger"), "stability_analyzer": ("config", "logger"),
                  "decision_maker": ("config", "logger", "stability_analyzer"), "car_analyzer": ("config", "logger")}
def _run_signature():
    return {"run_mode": RUN_MODE, "seed": SEED, "samples": SAMPLES_PER_DATASET, "max_cycles": MAX_CYCLES,
            "max_new_tokens": MAX_NEW_TOKENS, "model": MODEL_NAME, "subset": SUBSET_PER_DOMAIN}

def _save_resume(orch, S, next_cycle, finished):
    eng = orch.rsi_engine
    try:
        state = {"signature": _run_signature(), "next_cycle": next_cycle, "finished": finished, "S": S,
                 "components": {n: {k: v for k, v in vars(getattr(eng, n)).items() if k not in ex}
                                for n, ex in _STATE_EXCLUDE.items()},
                 "thresholds": orch.config.rsi.learned_thresholds,
                 "np_rng": np.random.get_state(), "torch_rng": torch.get_rng_state(),
                 "cycle_timings": RUN_STATE["cycle_timings"],
                 "sessions": PREV_SESSIONS + [{"started": MANIFEST["started_at"], "precision": RUN_STATE["precision_effective"],
                                               "hours": round((time.time() - _RUN_T0) / 3600, 2)}]}
        p = Path(orch.config.output_dir) / "resume_state.pkl"
        with open(p.with_suffix(".tmp"), "wb") as f:
            pickle.dump(state, f)
        os.replace(p.with_suffix(".tmp"), p)
    except Exception as e:
        print(f"⚠️ Không lưu được resume_state.pkl: {type(e).__name__}: {e}")

RESUME_STATE, PREV_SESSIONS = None, []
_rp = (RESUME if RESUME != "auto" else
       (sorted(glob.glob("/kaggle/input/**/resume_state.pkl", recursive=True), key=os.path.getmtime) or [None])[-1]) if RESUME else None
if _rp:
    with open(_rp, "rb") as f:
        _st = pickle.load(f)
    if _st.get("finished"):
        print(f"resume_state.pkl ở {_rp} là của lần chạy đã hoàn tất -> bỏ qua, chạy mới.")
    elif _st["signature"] != _run_signature():
        raise RuntimeError(f"resume_state.pkl không khớp cấu hình hiện tại:\n  file: {_st['signature']}\n  hiện tại: {_run_signature()}")
    else:
        RESUME_STATE, PREV_SESSIONS = _st, _st.get("sessions", [])
        print(f"↺ Sẽ chạy tiếp từ vòng {_st['next_cycle']} (phiên thứ {len(PREV_SESSIONS) + 1}), "
              f"còn {sum(not v['done'] for v in _st['S'].values())} task.")

# ---------------------------------------------------------------- (c) hiệu chỉnh ngưỡng theo lô
def _lockstep_calibration(self, calibration_tasks):
    """Như RSIEngine.run_calibration gốc (5 task × 1 câu đầu + 3 vòng), sinh theo lô."""
    if RESUME_STATE is not None:        # phiên tiếp nối: dùng lại đúng ngưỡng + trạng thái của phiên trước
        for name, d in RESUME_STATE["components"].items():
            vars(getattr(self, name)).update(d)
        self.config.rsi.learned_thresholds = RESUME_STATE["thresholds"]
        np.random.set_state(RESUME_STATE["np_rng"]); torch.set_rng_state(RESUME_STATE["torch_rng"])
        self.logger.info(f"↺ Resume: bỏ qua calibration, khôi phục ngưỡng và trạng thái (tiếp từ vòng {RESUME_STATE['next_cycle']})")
        return
    self.logger.info("Running calibration phase (batched)...")
    ctasks = calibration_tasks[:5]
    prefix = list(self.drift_detector._calibration_drifts)
    gens = batched_generate(self.model, [self.prompt_builder.build_initial_prompt(t) for t in ctasks])
    responses = [g[0] for g in gens]
    for t, r, e in zip(ctasks, responses, batched_embeddings(self.model, responses)):
        self.drift_detector.initialize_baseline(t.task_id, r, e)
    records = []
    for cycle in range(3):
        prompts = [self.prompt_builder.build_improvement_prompt(
                       t, r, self.quality_evaluator.evaluate(t, r),
                       DriftMeasurement(0, 0, 0, 0, 0, DriftSeverityLevel.NOMINAL, (0, 0)), 1.0, [], cycle + 1)
                   for t, r in zip(ctasks, responses)]
        responses = [g[0] for g in batched_generate(self.model, prompts)]
        for k, (t, r, e) in enumerate(zip(ctasks, responses, batched_embeddings(self.model, responses))):
            n0 = len(self.drift_detector._calibration_drifts)
            self.drift_detector.compute_drift(t.task_id, r, e)
            records.append((k, cycle, self.drift_detector._calibration_drifts[n0:]))
    # Khôi phục thứ tự task-trước-vòng-sau như vòng lặp tuần tự gốc (np.diff trong calibrate_from_data phụ thuộc thứ tự)
    self.drift_detector._calibration_drifts = prefix + [v for _, _, vs in sorted(records, key=lambda x: x[:2]) for v in vs]
    self.drift_detector.calibrate_thresholds()
    self.logger.info("✓ Calibration complete")

RSIEngine.run_calibration = _lockstep_calibration

# ---------------------------------------------------------------- (c2) tập con task cho one_session
_ORIG_LOAD_ALL = DatasetLoader.load_all
def _load_all_subset(self):
    """Lấy mẫu đúng như tác giả (63 task/miền, cùng seed), rồi giữ SUBSET_PER_DOMAIN task mỗi miền.
    2 task đầu (task dùng để calibration) luôn được giữ, phần còn lại chọn ngẫu nhiên có seed; giữ nguyên thứ tự gốc."""
    datasets = _ORIG_LOAD_ALL(self)
    if not SUBSET_PER_DOMAIN:
        return datasets
    rng = np.random.RandomState(SEED + 1)
    for tt, ts in datasets.items():
        k = min(SUBSET_PER_DOMAIN, len(ts))
        keep = set(range(2)) | set(rng.choice(np.arange(2, len(ts)), size=k - 2, replace=False).tolist())
        datasets[tt] = [t for i, t in enumerate(ts) if i in keep]
    self.logger.info(f"✓ one_session: giữ {SUBSET_PER_DOMAIN} task/miền từ {self.config.dataset.samples_per_dataset} task "
                     f"của tác giả (tổng {sum(len(v) for v in datasets.values())})")
    return datasets
DatasetLoader.load_all = _load_all_subset

# ---------------------------------------------------------------- (d) vòng RSI theo lô (mọi miền cùng lúc)
def _lockstep_run_experiments(self):
    """Mỗi task đi đúng các bước của RSIEngine.run_single_task gốc; chỉ khác là các task cùng vòng được sinh chung lô.
    Chạy cả 3 miền song song để nếu chạm TIME_BUDGET_HOURS thì cả 3 miền bị cắt đều nhau."""
    self.logger.info("\n" + "=" * 60); self.logger.info("MAIN EXPERIMENT PHASE (batched lockstep)"); self.logger.info("=" * 60)
    eng, start = self.rsi_engine, datetime.now()
    tasks = [t for ts in self.datasets.values() for t in ts]
    if RESUME_STATE is not None:
        S, start_cycle = RESUME_STATE["S"], RESUME_STATE["next_cycle"]
        missing = [t.task_id for t in tasks if t.task_id not in S]
        if missing:
            raise RuntimeError(f"Bộ task khác phiên trước (thiếu {missing[:3]}…) — kiểm tra RUN_MODE/SEED")
        RUN_STATE["cycle_timings"] = RESUME_STATE.get("cycle_timings", []) + RUN_STATE["cycle_timings"]
    else:
        S = {t.task_id: {"results": [], "resp": None, "q": None, "d": None, "done": False} for t in tasks}
        start_cycle = 0
    ckpt = Path(self.config.output_dir) / "cycles_checkpoint.jsonl"
    finished, cycle = True, start_cycle
    for cycle in range(start_cycle, self.config.rsi.max_improvement_cycles):
        active = [t for t in tasks if not S[t.task_id]["done"]]
        if not active:
            break
        # Dự báo thời gian vòng tới từ vòng trước (tỉ lệ theo số task còn chạy) để không vượt giới hạn 12 giờ giữa chừng
        last = RUN_STATE["cycle_timings"][-1] if RUN_STATE["cycle_timings"] else None
        est_h = last["seconds"] * len(active) / max(last["active"], 1) / 3600 if last else 0.0
        if (time.time() - _RUN_T0) / 3600 + est_h > TIME_BUDGET_HOURS:
            finished = False
            RUN_STATE.update(budget_hit=True, budget_cut_cycle=cycle, budget_cut_tasks=[t.task_id for t in active])
            self.logger.warning(f"⏱ Vượt TIME_BUDGET_HOURS={TIME_BUDGET_HOURS}: dừng trước vòng {cycle}, "
                                f"{len(active)} task chưa xong. Trạng thái đã lưu ở resume_state.pkl — "
                                f"thêm Output phiên này làm Input rồi chạy lại để tiếp tục.")
            break
        c0 = time.time()
        prompts = []
        for t in active:
            s = S[t.task_id]
            if cycle == 0:
                prompts.append(eng.prompt_builder.build_initial_prompt(t))
            else:
                prompts.append(eng.prompt_builder.build_improvement_prompt(
                    t, s["resp"], s["q"], s["d"], s["results"][-1].constraint_preservation_score,
                    s["results"][-1].violations, cycle))
        gens = batched_generate(eng.model, prompts)
        embs = batched_embeddings(eng.model, [g[0] for g in gens])
        gen_share = (time.time() - c0) / len(active)
        with open(ckpt, "a", encoding="utf-8") as fck:
            for t, (response, gen_meta), embedding in zip(active, gens, embs):
                s, cycle_start = S[t.task_id], datetime.now()
                if cycle == 0:
                    eng.drift_detector.initialize_baseline(t.task_id, response, embedding)
                    drift = DriftMeasurement(goal_drift_index=0.0, semantic_drift=0.0, lexical_drift=0.0,
                                             structural_drift=0.0, distributional_drift=0.0,
                                             severity=DriftSeverityLevel.NOMINAL, confidence_interval=(0.0, 0.0))
                else:
                    drift = eng.drift_detector.compute_drift(t.task_id, response, embedding)
                cps, violations = eng.constraint_evaluator.evaluate(t, response)
                quality = eng.quality_evaluator.evaluate(t, response)
                car = eng.car_analyzer.compute_car(quality.aggregate_score, cps, drift.goal_drift_index)
                eng.stability_analyzer.record_cycle(t.task_id, cycle, quality.aggregate_score, drift.goal_drift_index, cps)
                regression_risk, _ = eng.stability_analyzer.compute_regression_risk(t.task_id, cycle)
                decision, reason = eng.decision_maker.decide(t.task_id, cycle, quality, drift, cps, regression_risk)
                result = CycleResult(
                    cycle_number=cycle, task_id=t.task_id, task_type=t.task_type.value, response=response,
                    response_length=len(response), tokens_generated=gen_meta["output_tokens"], quality=quality,
                    drift=drift, constraint_preservation_score=cps, violations=violations,
                    capability_alignment_ratio=car, regression_risk=regression_risk, decision=decision,
                    decision_reason=reason,
                    generation_time_seconds=gen_share + (datetime.now() - cycle_start).total_seconds())
                s["results"].append(result)
                eng.logger.log_cycle_summary(cycle=cycle, task_id=t.task_id, task_type=t.task_type.value,
                                             quality=quality.aggregate_score, drift=drift.goal_drift_index, cps=cps,
                                             violations=len(violations), decision=f"{decision}:{reason}")
                fck.write(json.dumps({**result.to_dict(), "response": response}, default=str, ensure_ascii=False) + "\n")
                if decision == "stop":
                    s["done"] = True
                else:
                    s["resp"], s["q"], s["d"] = response, quality, drift
        dt = time.time() - c0
        RUN_STATE["cycle_timings"].append({"cycle": cycle, "active": len(active), "seconds": round(dt, 1),
                                           "batch": _BATCH["gen"]})
        print(f"▶ Vòng {cycle:02d}: {len(active):3d} task | {dt/60:5.1f} phút | "
              f"đã chạy {(time.time() - _RUN_T0)/3600:4.2f} giờ | còn {sum(not v['done'] for v in S.values())} task", flush=True)
        _save_resume(self, S, cycle + 1, finished=False)
    RUN_STATE["finished"] = finished
    _save_resume(self, S, cycle + 1 if finished else cycle, finished=finished)
    for task_type, ts in self.datasets.items():
        results = {t.task_id: S[t.task_id]["results"] for t in ts}
        eng.results.update(results)
        self.all_results[task_type] = results
        if self.config.save_intermediate_results:
            self._save_intermediate(task_type, results)
    self.logger.info(f"\n✓ Experiments completed in {(datetime.now() - start).total_seconds() / 60:.1f} minutes")
    return self.all_results

ExperimentOrchestrator.run_experiments = _lockstep_run_experiments

# ---------------------------------------------------------------- (e) số vòng tối thiểu + kiểm định an toàn (như v1)
rsi_params = dict(vars(config.rsi)) if hasattr(config.rsi, "__dict__") else {}
for k, v in rsi_params.items():
    if "min" in k.lower() and "cycle" in k.lower() and isinstance(v, int) and not isinstance(v, bool) \
            and config.rsi.max_improvement_cycles <= v:
        print(f"⚠️ {k}={v} >= max_improvement_cycles; tăng lên {v + 1}.")
        config.rsi.max_improvement_cycles = v + 1
EFFECTIVE_MAX_CYCLES = config.rsi.max_improvement_cycles

_NanResult = namedtuple("NanResult", ["statistic", "pvalue"])
def _make_safe(fn, name):
    def _safe(*a, **k):
        try:
            return fn(*a, **k)
        except (ValueError, ZeroDivisionError, FloatingPointError) as e:
            print(f"[bỏ qua {name}] {e}")
            return _NanResult(float("nan"), float("nan"))
    _safe._sahoo_safe = True     # (v1 dùng __wrapped__ làm dấu, nhưng hàm scipy vốn đã có __wrapped__ -> bản vá bị bỏ qua)
    return _safe
for _name in ["kruskal", "wilcoxon", "friedmanchisquare", "shapiro", "levene", "f_oneway", "mannwhitneyu", "ttest_1samp"]:
    if hasattr(stats, _name) and not getattr(getattr(stats, _name), "_sahoo_safe", False):
        _safe_fn = _make_safe(getattr(stats, _name), _name)
        setattr(stats, _name, _safe_fn)
        if _name in globals():
            globals()[_name] = _safe_fn

# ---------------------------------------------------------------- (f) manifest
config.save()
MANIFEST = {
    "notebook_version": "v2", "run_mode": RUN_MODE, "seed": SEED, "model": config.model.name,
    "precision_requested": PRECISION, "dtype": config.model.torch_dtype, "thinking": ENABLE_THINKING,
    "samples_per_dataset": config.dataset.samples_per_dataset, "max_cycles_requested": MAX_CYCLES,
    "max_cycles_effective": EFFECTIVE_MAX_CYCLES, "max_new_tokens": config.model.max_new_tokens,
    "temperature": config.model.temperature, "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "gen_batch_size": GEN_BATCH_SIZE, "time_budget_hours": TIME_BUDGET_HOURS,
    "gpu": f"{gpu_name} x{torch.cuda.device_count()}", "python": sys.version.split()[0],
    "torch": torch.__version__, "transformers": transformers.__version__,
    "output_dir": config.output_dir, "started_at": datetime.now().isoformat(timespec="seconds"),
}
print(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
gc.collect(); torch.cuda.empty_cache()

## Chạy thực nghiệm
Vòng RSI được sinh theo lô. Mỗi vòng in một dòng `▶ Vòng …` kèm thời gian đã chạy, để theo dõi trong log của phiên commit.

In [ ]:
# CELL 5 — Chạy thực nghiệm (calibration -> các vòng RSI theo lô -> thống kê -> biểu đồ)
OUTPUT_DIR = config.output_dir
experiment_output, RUN_ERROR = None, None
_RUN_T0 = time.time()
try:
    experiment_output = run_experiment()
    if isinstance(experiment_output, dict) and experiment_output.get("output_dir"):
        OUTPUT_DIR = experiment_output["output_dir"]
except Exception as e:
    import traceback
    RUN_ERROR = f"{type(e).__name__}: {e}"
    traceback.print_exc()
    print(f"\n⚠️ run_experiment dừng với lỗi: {RUN_ERROR}\nKết quả đã lưu (nếu có, gồm cycles_checkpoint.jsonl) ở: {OUTPUT_DIR}")
finally:
    try:
        thr = config.rsi.learned_thresholds
        MANIFEST["calibrated_thresholds"] = {lvl.name: float(v) for lvl, v in thr.drift_thresholds.items()}
        MANIFEST["convergence_threshold"] = float(thr.convergence_threshold)
        MANIFEST["regression_tolerance"] = float(thr.regression_tolerance)
    except Exception:
        pass
    MANIFEST.update({k: v for k, v in RUN_STATE.items() if k != "budget_cut_tasks"})
    MANIFEST["budget_cut_task_count"] = len(RUN_STATE["budget_cut_tasks"])
    MANIFEST["gen_batch_size_final"] = _BATCH["gen"]
    MANIFEST["runtime_minutes"] = round((time.time() - _RUN_T0) / 60, 1)
    MANIFEST["finished_at"] = datetime.now().isoformat(timespec="seconds")
    MANIFEST["run_error"] = RUN_ERROR
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    with open(Path(OUTPUT_DIR) / "kaggle_run_manifest.json", "w", encoding="utf-8") as f:
        json.dump(MANIFEST, f, indent=2, ensure_ascii=False, default=str)
    print(f"Thời gian chạy: {MANIFEST['runtime_minutes']} phút | precision: {RUN_STATE['precision_effective']} "
          f"| chạm giới hạn thời gian: {RUN_STATE['budget_hit']}")

# Ước lượng thời gian cho các profile lớn (dùng sau lần smoke)
_ct = RUN_STATE["cycle_timings"]
if _ct:
    per_batch = np.median([c["seconds"] / math.ceil(c["active"] / max(c["batch"], 1)) for c in _ct])
    print("\nƯớc lượng thời gian vòng RSI (chưa tính ~15–25 phút tải mô hình + calibration + phân tích):")
    for name, prof in PROFILES.items():
        n_tasks = 3 * (prof["subset"] or prof["samples"])
        worst = prof["cycles"] * math.ceil(n_tasks / GEN_BATCH_SIZE) * per_batch * 1.2 / 3600
        print(f"  {name:8s}: tệ nhất ≈ {worst:4.1f} giờ | thường ≈ {0.6*worst:4.1f} giờ (nhiều task dừng sớm)")
    print(f"  -> Mỗi phiên chạy tối đa ~{TIME_BUDGET_HOURS} giờ; số phiên cần ≈ ước lượng / {TIME_BUDGET_HOURS} (làm tròn lên). "
          f"Notebook tự lưu trạng thái và chạy tiếp ở phiên sau.")

In [ ]:
# CELL 6 — Phân tích hậu kiểm của tác giả (không chạy lại ablation tốn GPU)
posthoc_results = None
try:
    posthoc_results = run_post_experiment_analysis(OUTPUT_DIR, run_ablations=False, run_theoretical=True)
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"⚠️ Post-hoc analysis lỗi: {type(e).__name__}: {e} (kết quả thí nghiệm chính không bị ảnh hưởng)")

## Tổng hợp kết quả và đối chiếu với bài báo
Cell 7 liệt kê file kết quả. Cell 8 tự tính cột "tái tạo" và đặt cạnh Table 1 của bài báo (arXiv 2603.06333).

In [ ]:
# CELL 7 — Liệt kê và xem nhanh các file kết quả
from IPython.display import display, Image as IPImage
out_dir = Path(OUTPUT_DIR)
files = sorted(p for p in out_dir.rglob("*") if p.is_file())
for p in files:
    print(f"{str(p.relative_to(out_dir)):55s} {p.stat().st_size/1024:9.1f} KB")

for p in files:
    try:
        if p.suffix == ".csv":
            df = pd.read_csv(p)
            print(f"\n=== {p.name}: {df.shape[0]} dòng × {df.shape[1]} cột ===")
            display(df.head(10))
        elif p.suffix == ".json" and p.stat().st_size < 200_000:
            obj = json.load(open(p, encoding="utf-8"))
            print(f"\n=== {p.name} ===")
            print(json.dumps(obj, indent=1, ensure_ascii=False, default=str)[:3000])
    except Exception as e:
        print(f"(không đọc được {p.name}: {e})")

for p in files:
    if p.suffix == ".png":
        print(f"\n--- {p.name} ---")
        display(IPImage(filename=str(p), width=900))

In [ ]:
# CELL 8 — Bảng đối chiếu với bài báo (tự tính cột "tái tạo")
if not RUN_STATE.get("finished", True):
    print("⚠️ KẾT QUẢ TỪNG PHẦN: lần chạy chưa xong (dừng vì giới hạn thời gian). Hãy chạy phiên tiếp theo; đừng dùng bảng này trong báo cáo.\n")
DOMAINS = [("code_generation", "Code"), ("truthfulness", "Truth"), ("mathematical_reasoning", "Math")]
OUR_IDS = {}
def _load_domain(dom):
    p = Path(OUTPUT_DIR) / f"results_{dom}.json"
    return json.load(open(p, encoding="utf-8")) if p.exists() else {}

try:
    crit = float(config.rsi.learned_thresholds.drift_thresholds[DriftSeverityLevel.CRITICAL])
except Exception:
    crit = np.nan

rows = {"Initial quality": [], "Final quality": [], "Δ quality (final − initial)": [], "p (Wilcoxon, initial vs final)": [],
        "Goal Drift Index (mean)": [], "GDI mean (bỏ vòng 0)": [], "Constraint preservation": [],
        "Capability-Alignment Ratio": [], "Mean cycles per task": [], "Critical GDI threshold": []}
stop_tbl = {}
for dom, short in DOMAINS:
    R = _load_domain(dom)
    OUR_IDS[dom] = set(R)
    runs = [v for v in R.values() if v]
    if not runs:
        for k in rows: rows[k].append(np.nan)
        continue
    q0 = np.array([r[0]["quality"]["aggregate"] for r in runs])
    qT = np.array([r[-1]["quality"]["aggregate"] for r in runs])
    allc = [c for r in runs for c in r]
    try:
        pval = stats.wilcoxon(q0, qT).pvalue if np.any(q0 != qT) else np.nan
    except Exception:
        pval = np.nan
    rows["Initial quality"].append(q0.mean()); rows["Final quality"].append(qT.mean())
    rows["Δ quality (final − initial)"].append(qT.mean() - q0.mean()); rows["p (Wilcoxon, initial vs final)"].append(pval)
    rows["Goal Drift Index (mean)"].append(np.mean([c["drift"]["goal_drift_index"] for c in allc]))
    rows["GDI mean (bỏ vòng 0)"].append(np.mean([c["drift"]["goal_drift_index"] for c in allc if c["cycle_number"] > 0]))
    rows["Constraint preservation"].append(np.mean([c["cps"] for c in allc]))
    rows["Capability-Alignment Ratio"].append(np.mean([c["car"] for c in allc]))
    rows["Mean cycles per task"].append(np.mean([len(r) for r in runs]))
    rows["Critical GDI threshold"].append(crit)
    stop_tbl[short] = pd.Series([r[-1]["decision_reason"] for r in runs]).value_counts()

paper_vals = {
    "Initial quality": [0.672, 0.678, 0.689], "Final quality": [0.795, 0.704, 0.805],
    "Goal Drift Index (mean)": [0.320, 0.354, 0.330], "Constraint preservation": [1.000, 0.987, 1.000],
    "Capability-Alignment Ratio": [0.671, 0.599, 0.675], "Mean cycles per task": [7.1, 10.4, 8.9],
    "Critical GDI threshold": [0.44, 0.44, 0.44],
}
cmp_rows = []
for k, vals in rows.items():
    pv = paper_vals.get(k, [np.nan] * 3)
    if k == "Δ quality (final − initial)":
        pv = [round(paper_vals["Final quality"][i] - paper_vals["Initial quality"][i], 3) for i in range(3)]
    cmp_rows.append([k, *pv, *vals])
# Kết quả gốc của tác giả trong repo (data/results/*.json) — cùng code, cùng cách tính
REPO_RAW = ("https://raw.githubusercontent.com/SubramanyamSahoo/SAHOO-Safeguarded-Alignment-for-High-Order-Optimization-"
            "Objectives-in-Recursive-Self-Improvement/main/data/results/results_{}.json")
repo_cols = []
for dom, _ in DOMAINS:
    try:
        _all = requests.get(REPO_RAW.format(dom), timeout=60).json()
        Rr = [v for k, v in _all.items() if v and (not SUBSET_PER_DOMAIN or k in OUR_IDS.get(dom, ()))]
        q0r = np.array([r[0]["quality"]["aggregate"] for r in Rr]); qTr = np.array([r[-1]["quality"]["aggregate"] for r in Rr])
        ac = [c for r in Rr for c in r]
        repo_cols.append({"Initial quality": q0r.mean(), "Final quality": qTr.mean(),
                          "Δ quality (final − initial)": qTr.mean() - q0r.mean(),
                          "Goal Drift Index (mean)": np.mean([c["drift"]["goal_drift_index"] for c in ac]),
                          "GDI mean (bỏ vòng 0)": np.mean([c["drift"]["goal_drift_index"] for c in ac if c["cycle_number"] > 0]),
                          "Constraint preservation": np.mean([c["cps"] for c in ac]),
                          "Capability-Alignment Ratio": np.mean([c["car"] for c in ac]),
                          "Mean cycles per task": np.mean([len(r) for r in Rr]), "Critical GDI threshold": 0.4434})
    except Exception as e:
        print(f"(không tải được kết quả repo cho {dom}: {e})"); repo_cols.append({})
for row in cmp_rows:
    row[4:4] = [repo_cols[i].get(row[0], np.nan) for i in range(3)]
comparison = pd.DataFrame(cmp_rows, columns=["Chỉ số", "Code (bài báo)", "Truth (bài báo)", "Math (bài báo)",
                                             "Code (repo)", "Truth (repo)", "Math (repo)",
                                             "Code (tái tạo)", "Truth (tái tạo)", "Math (tái tạo)"])
print("Cột (repo) = kết quả gốc tác giả công bố trong repo, cùng code với bản tái tạo"
      + (f", tính trên ĐÚNG {SUBSET_PER_DOMAIN} task/miền mà lần tái tạo này chạy" if SUBSET_PER_DOMAIN else "")
      + "; ngưỡng critical lấy từ logs/experiment.log.")
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("Ghi chú: 'Mean cycles per task' = số vòng trung bình mỗi task chạy trước khi dừng (bài báo gọi là mean convergence cycle).")
display(comparison)
print("\nLý do dừng ở vòng cuối của mỗi task:")
display(pd.DataFrame(stop_tbl).fillna(0).astype(int))

deviations = pd.DataFrame({
    "Thiết lập": ["Mô hình / độ chính xác", "Số mẫu/miền", "Số vòng tối đa", "Max new tokens", "Thinking mode",
                  "Bootstrap", "Phần cứng", "Cách sinh", "Bị cắt do giới hạn thời gian"],
    "Repo tác giả (config.json)": ["Qwen3-8B (bf16)", 63, 20, 2048, "mặc định (không tắt)", 2000, "không nêu (code nhắm H100)", "tuần tự", "—"],
    "Tái tạo v2": [f"Qwen3-8B ({RUN_STATE['precision_effective']})",
                   (f"{SUBSET_PER_DOMAIN} (tập con của đúng 63 task tác giả)" if SUBSET_PER_DOMAIN else config.dataset.samples_per_dataset),
                   EFFECTIVE_MAX_CYCLES, config.model.max_new_tokens, "tắt" if not ENABLE_THINKING else "bật",
                   BOOTSTRAP_SAMPLES, MANIFEST["gpu"], f"theo lô (batch {GEN_BATCH_SIZE}), logic từng task giữ nguyên",
                   (f"có: dừng trước vòng {RUN_STATE['budget_cut_cycle']}, {len(RUN_STATE['budget_cut_tasks'])} task"
                    if RUN_STATE["budget_hit"] else "không")],
})
display(deviations)
comparison.to_csv(Path(OUTPUT_DIR) / "paper_vs_reproduction.csv", index=False)
deviations.to_csv(Path(OUTPUT_DIR) / "setup_deviations.csv", index=False)

In [ ]:
# CELL 9 — Nén kết quả để tải về (tab Output của Kaggle)
import shutil
_tag = "final" if RUN_STATE.get("finished", True) else f"partial-c{RUN_STATE['budget_cut_cycle']}"
zip_base = WORK_DIR / f"sahoo_v2_{RUN_MODE}_seed{SEED}_{RUN_STATE['precision_effective']}_{_tag}_{datetime.now():%Y%m%d_%H%M}"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("Đã nén:", zip_path, f"({Path(zip_path).stat().st_size/1024/1024:.1f} MB)")

## Checklist sau mỗi lần chạy
- **Bước 1 `smoke`:**
  - Log phải có dòng `Model loaded: … precision=fp16 … batched_embedding=True`.
    Nếu thấy `precision=nf4`, fp16 đã bị tràn số và notebook tự lùi về 4-bit; vẫn chạy được, nhưng phải ghi rõ trong báo cáo.
  - Có các dòng `▶ Vòng 00 … ▶ Vòng 03` và giá trị `D:` khác 0 từ vòng 1.
  - Cuối Cell 5 có ước lượng giờ cho `paper` và `reduced`. Hãy chọn profile cho bước 2 theo ước lượng này.
- **Bước 2 `one_session`:** log phải có `✓ one_session: giữ 30 task/miền …`, cuối cùng ra file `…_final_…zip`.
- **(Chỉ khi chạy `paper`):**
  - Nếu log báo `⏱ Vượt TIME_BUDGET_HOURS`, đó là bình thường: mở phiên mới, *Add Input → Notebook output* của phiên vừa xong,
    giữ nguyên `RUN_MODE`/`SEED` rồi Save & Run All. Đầu Cell 4 phải in `↺ Sẽ chạy tiếp từ vòng …`.
  - Kết quả cuối lấy từ phiên có manifest `"finished": true`.
- **Báo cáo:**
  - Đưa vào `paper_vs_reproduction.csv`, `setup_deviations.csv` và manifest (precision thực tế, ngưỡng đã hiệu chỉnh, batch size cuối).
  - Ngưỡng critical được hiệu chỉnh lại từ dữ liệu (median + 4·MAD), nên có thể khác 0.44 của bài báo. Báo cáo cả hai giá trị.
- Nếu phiên bị ngắt giữa chừng, `cycles_checkpoint.jsonl` vẫn chứa mọi vòng đã chạy xong.